Lecture 4 transitions from evaluating fixed policies to **model-free control**, where the agent learns the optimal policy through direct experience without a model of the world. A major highlight is the development of **Deep Q-Networks (DQN)**, which allowed agents to learn complex tasks like Atari games directly from pixel inputs.

### **1. The Exploration-Exploitation Trade-off**
A central challenge in reinforcement learning is balancing **exploration** (trying new actions to gather data) and **exploitation** (using current knowledge to maximize reward).
*   **The Coverage Problem:** Deterministic policies only take one action per state, meaning the agent never learns about alternative actions.
*   **Epsilon-greedy ($\epsilon$-greedy):** A simple solution where the agent acts greedily with probability $1-\epsilon$ and chooses a random action with probability $\epsilon$. This strategy ensures the agent continues to try multiple actions in every state.
*   **GLIE (Greedy in the Limit of Infinite Exploration):** A condition for convergence where all state-action pairs are visited infinitely often, and the policy eventually becomes greedy.

### **2. Model-Free Control Algorithms**
The lecture details three primary methods for learning optimal policies in a tabular setting:
*   **Monte Carlo (MC) Control:** An **on-policy** method that updates Q-values after full episodes by averaging observed returns. 
*   **SARSA (State-Action-Reward-State-Action):** An **on-policy** Temporal Difference (TD) method. It updates the Q-value of the current state-action pair using the **actual next action** taken by the policy.
*   **Q-Learning:** An **off-policy** TD method. Unlike SARSA, it updates the Q-value using the **maximum possible reward** from the next state, regardless of which action the agent actually takes.

### **3. Function Approximation and the "Deadly Triad"**
To handle massive state spaces like image pixels, agents use **function approximation** (e.g., deep neural networks) rather than tables to represent Q-functions. This allows for **generalization**, where the agent applies what it learns in one state to similar unseen states. However, the lecture warns of the **"Deadly Triad"**, a combination of three factors that can cause learning to oscillate or diverge:
1.  **Bootstrapping:** Updating estimates based on other estimates (TD learning).
2.  **Function Approximation:** Using non-linear models like neural networks.
3.  **Off-policy Learning:** Learning from data gathered by a different policy (Q-learning).

### **4. Deep Q-Networks (DQN)**
DQN stabilized deep reinforcement learning through two key innovations that addressed the instabilities of the "Deadly Triad":
*   **Experience Replay:** Transitions are stored in a **replay buffer** and sampled randomly for training. This breaks correlations between consecutive samples and improves **data efficiency** by reusing past data.
*   **Fixed Q-Targets:** The agent maintains two sets of weights. One set is updated frequently, while a separate set of **target weights** is used to calculate the target values for updates and is only updated periodically. This prevents the target from moving too quickly and makes training more like stable supervised learning. 

While keeping two sets of weights **doubles the memory requirements**, it significantly improves stability and performance across various tasks.

This coding exercise is designed to reinforce the concepts of **model-free control**, **function approximation**, and the stabilizing mechanisms of **Deep Q-Networks (DQN)** as discussed in the sources.

### **Environment Setup**
Assume we are working with a standard MDP where:
*   `s`: Current state.
*   `a`: Action taken.
*   `r`: Reward received.
*   `s_prime`: Next state reached.
*   `a_prime`: Next action chosen by the current policy.
*   `gamma`: Discount factor ($\gamma$).
*   `alpha`: Learning rate ($\alpha$).

---

### **Part 1: Tabular Model-Free Control**
In tabular land, we store values in a 2D array `Q[state, action]`. These algorithms use **$\epsilon$-greedy exploration** to balance exploring the environment and exploiting current knowledge.

#### **1.1 Monte Carlo (MC) Control (On-Policy)**
MC control updates values based on the total return ($G$) observed at the end of an episode.

In [ ]:
def update_mc_control(Q, episode, alpha, gamma):
    """
    episode: list of (s, a, r) tuples
    """
    G = 0
    # Process the episode backwards to calculate returns
    for s, a, r in reversed(episode):
        # TODO: Update G (the discounted return)
        G = r + gamma * G
        # TODO: Update Q[s, a] using the incremental update formula 
        # Hint: Q = Q + alpha * (Target - Q)
        Q[s][a] += alpha * (G - Q[s][a])
    return Q

#### **1.2 SARSA (On-Policy TD)**
SARSA updates the Q-value after every step using the **actual next action** ($a'$) taken by the current policy.

In [ ]:
def update_sarsa(Q, s, a, r, s_prime, a_prime, alpha, gamma):
    # TODO: Define the SARSA Target 
    # Hint: Use the actual next action a_prime
    target = r + gamma * Q[s_prime][a_prime]
    
    # TODO: Update Q[s, a]
    Q[s][a] += alpha * (target - Q[s][a])
    return Q

#### **1.3 Q-Learning (Off-Policy TD)**
Q-learning updates the Q-value using the **maximum possible reward** from the next state, regardless of what action was actually taken.

In [ ]:
def update_q_learning(Q, s, a, r, s_prime, alpha, gamma):
    # TODO: Define the Q-Learning Target 
    # Hint: Use the 'max' operator over all possible actions in s_prime
    target = r + gamma * max(Q[s_prime].values())
    
    # TODO: Update Q[s, a]
    Q[s][a] += alpha * (target - Q[s][a])
    return Q

### **Part 2: Linear Function Approximation (Revised)**

In this section, you will transition from a tabular representation to **function approximation**. Instead of a table, we represent the Q-function as a parameterized linear function: $Q(s, a; W) = \phi(s, a)^T W$, where $W$ is a weight vector and $\phi(s, a)$ is a feature vector. 

To update these weights, we use the **semi-gradient** update rule. This method adjusts the weights to minimize the error between the current prediction and a target value (such as a TD target or a Monte Carlo return).

**Instructions:**
*   Assume that the weight vector `W` and feature vectors are represented as **NumPy arrays**.
*   You are **provided with a function** `feature_func(s, a)` that takes a state and an action as input and returns a NumPy array representing the feature vector $\phi(s, a)$.
*   Implement the weight update logic below.

In [ ]:
import numpy as np

def semi_gradient_q_update(W, s, a, target, alpha, feature_func):
    """
    Performs a semi-gradient update for a linear Q-function approximator.
    
    Args:
        W: A NumPy array representing the current weight vector.
        s: The current state.
        a: The action taken.
        target: The target value (scalar), such as a TD target or MC return.
        alpha: The learning rate (scalar).
        feature_func: A provided function that returns a NumPy array phi(s, a).
        
    Returns:
        The updated weight vector W as a NumPy array.
    """
    # 1. Retrieve the feature vector phi(s, a) using the provided feature_func
    # Note: In a linear model, this feature vector is also the gradient of Q(s,a) wrt W.
    phi = feature_func(s, a)
    
    # 2. Calculate the current prediction: Q(s, a; W) = phi(s, a) dot W
    current_prediction = np.dot(phi, W)
    
    # 3. Compute the update to the weights W
    # Hint: W = W + alpha * (target - current_prediction) * gradient
    W += alpha * (target - current_prediction) * phi
    # TODO: Implement the logic and return the new weight vector
    return W


### **Conceptual Review**
*   **The Gradient:** Why is the feature vector $\phi(s, a)$ used as the gradient in this linear update?
*   **Generalization:** How does updating a single weight vector $W$ allow the agent to generalize its learning to states it hasn't visited yet, compared to the tabular methods in Part 1?
*   **The "Deadly Triad":** Why is this specific type of update (function approximation + bootstrapping) considered potentially unstable?

### **Part 3: Deep Q-Networks (DQN) (Revised)**

In this section, you will implement the stabilizing mechanisms that allowed Deep Q-Networks to overcome the **"Deadly Triad"** (bootstrapping, function approximation, and off-policy learning). While tabular methods and linear approximation can converge under specific conditions, **DQN is not guaranteed to converge** to the optimal Q-function due to potential instabilities and "realizability" issues—where the functional form of the neural network might not be able to perfectly represent the true Q-function.

**Instructions:**
*   Assume all state transitions and batches are handled as **NumPy arrays**.
*   Assume you are provided with two function approximators (neural networks): **`local_net(s)`** and **`target_net(s)`**. Both take a NumPy array of states and return a NumPy array of Q-values for all actions.
*   The **`local_net`** is updated frequently, while the **`target_net`** provides stable "Fixed Q-Targets" [DQN Summary].

---

#### **3.1 The Experience Replay Buffer**
To break the correlations between consecutive samples and improve data efficiency, we store transitions in a buffer and sample random mini-batches for training [DQN Summary].

In [ ]:
import numpy as np

class ReplayBuffer:
    def __init__(self, capacity):
        # Initialize the buffer to store transitions as NumPy arrays
        self.capacity = capacity
        self.buffer = []
        
    def push(self, s, a, r, s_prime, done):
        """
        Args:
            s, s_prime: NumPy arrays representing states.
            a, r, done: Scalars (action, reward, terminal flag).
        """
        # TODO: Add the transition tuple to the buffer
        # If the buffer exceeds capacity, remove the oldest entry
        if len(self.buffer) >= self.capacity:
            self.buffer.pop(0)
        self.buffer.append((s, a, r, s_prime, done))

    def sample(self, batch_size):
        """
        Returns:
            A mini-batch of transitions as a set of NumPy arrays.
        """
        # TODO: Randomly sample 'batch_size' transitions from the buffer
        # Return them as (states, actions, rewards, next_states, dones)
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[i] for i in indices])
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)

---

#### **3.2 DQN Training Step (Fixed Q-Targets)**
The key to DQN's stability is using a separate **target network** to compute the "Y" targets, which prevents the goalposts from moving too quickly during training [DQN Summary].

In [ ]:
def compute_dqn_loss(local_net, target_net, batch, gamma):
    """
    Args:
        local_net: The 'live' network used for making current predictions.
        target_net: The 'frozen' network used to calculate stable targets.
        batch: A tuple of NumPy arrays (states, actions, rewards, next_states, dones).
        gamma: The discount factor (scalar).
    """
    states, actions, rewards, next_states, dones = batch

    # 1. Get Q-values for all actions in 'states' from the local_net
    current_q_values = local_net(states)
    # 2. Extract the Q-values for the specific 'actions' taken (Current Predictions)
    current_predictions = current_q_values[np.arange(len(actions)), actions]

    # 3. Get Q-values for all actions in 'next_states' from the target_net
    next_q_values = target_net(next_states)
    # 4. Use the 'max' operator to find the best Q-value for each next_state (Off-policy)
    max_next_q_values = np.max(next_q_values, axis=1)

    # 5. Calculate the DQN Target (Y):
    # Hint: Target = rewards + gamma * max_Q_next_state * (1 - dones)
    # The (1 - dones) term ensures the future value is 0 if the episode ended.
    targets = rewards + gamma * max_next_q_values * (1 - dones)
    
    # TODO: Implement the logic to compute the 'error' or 'loss' 
    # (the difference between the Target and the Current Predictions)
    prediction_errors = targets - current_predictions
    return prediction_errors

---

### **Conceptual Checklist (To Review After Solving)**
1.  **Exploration:** Why is an **$\epsilon$-greedy policy** necessary to ensure "coverage" of the state-action space?
2.  **Comparison:** How does the target for **SARSA** differ from the target for **Q-Learning**?
3.  **DQN Stability:** Why does keeping a separate set of **target weights** ($W^-$) make reinforcement learning more like stable supervised learning?
4.  **Performance:** According to the source, which innovation—**Experience Replay** or **Fixed Targets**—provided the single largest boost in performance for the Atari agents?